prepatring for model

In [1]:
import pandas as pd
real_data = pd.read_csv('../datasets/pvAmpSeq_simulation_ready.csv')
real_data.head()

,patient_id,sample_id,episode,marker,allele,frequency,country,MOIs
0,Sero-040,Sero-040-R01,1.0,Chr01,Chr01-1,0.531524,ethiopia,6
1,Sero-182,Sero-182-R12,3.0,Chr11,Chr11-2,0.149320,ethiopia,3
2,Sero-182,Sero-182-R12,3.0,Chr13,Chr13-1,0.729045,ethiopia,3
3,Sero-182,Sero-182-R12,3.0,Chr13,Chr13-2,0.129575,ethiopia,3
4,Sero-182,Sero-182-R12,3.0,Chr14,Chr14-1,0.931269,ethiopia,3


In [7]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from DeepSets import DataPreprocessor, build_deepsets_tensors


def run_Prediction(data, mode='prediction'):

    # perform basic preprocess
    preprocessed_tensor = (
        data
        .pipe(DataPreprocessor.removing_single_episode_patients)
        .pipe(DataPreprocessor.calculate_population_Allele_frequencies, population='country')
        .pipe(DataPreprocessor.calculate_MOIs)
        .pipe(DataPreprocessor.paired_episode_assigner)
        .pipe(DataPreprocessor.build_deepsets_tensors, mode=mode)
    )
    loaded_data = DataPreprocessor.load_tensor(preprocessed_tensor)
    return loaded_data

result = run_Prediction(real_data)

11
394
5


In [3]:
import json
# reading allele dictionary
with open('allele_dict.json', 'r') as file:
    allele_to_id = json.load(file)

In [4]:
import torch
from DeepSets import PvDeepSets
# 1. Define the device
device = torch.device("cpu")

# 2. Initialize the model with the same parameters used during training
# Make sure len(allele_to_id) matches the training setup!
n_alleles = len(allele_to_id) 
model = PvDeepSets(n_alleles=n_alleles)
print(n_alleles)
# 3. Load the weights (state_dict)
# Replace "pv_deepsets_weights.pth" with your actual filename
model.load_state_dict(torch.load("../src/DeepSets/pv_deepsets_weights20260402.pth", map_location=device))



143


C:\Users\sinel\AppData\Local\Temp\ipykernel_24916\2347922982.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("../src/DeepSets/pv_deepse

<All keys matched successfully>

In [5]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

def get_model_results(model, dataloader, device, threshold=0.5):
    model.eval()
    all_predicted = []
    y_pred_filtered = []
    undetermined_count = 0

    with torch.no_grad():
        for X_alleles, allele_mask, marker_mask, priors, MOI in dataloader:
            # ... (your existing GPU transfer code) ...
            y_pred = model(X_alleles, allele_mask, marker_mask, priors, MOI)
            
            # Apply Softmax if your model doesn't already output probabilities
            # probs = torch.softmax(y_pred, dim=1) 
            probs = y_pred 

            # Get max probabilities and their indices
            max_probs, preds = torch.max(probs, dim=1)

            for i in range(len(max_probs)):
                if max_probs[i] >= threshold:
                    y_pred_filtered.append(preds[i].item())
                else:
                    undetermined_count += 1

            all_predicted.append(y_pred.cpu().numpy())

    print(f"Total undetermined samples removed: {undetermined_count}")
    return np.vstack(all_predicted), y_pred_filtered

predicted_probs, y_pred_hard = get_model_results(model, result, device)




Total undetermined samples removed: 1


In [17]:
predicted_probs[1].sum()

np.float32(1.0)

In [ ]:
# Step 1: removing patients with only one episode
# Step 2: make all possible pair (including making episode 1&2)
# step 3: Calculate MOIs
# step 4: Calculate Afreq